# Pipeline 155 — native 9-run continuation

The read-only cells preview one allocator/cohort and print the original results and winners. The final cell checks the entire registered plan and runs only eligible missing ninth positions. Runner/model checkpoints and logs go to Git; evaluators go to Pipeline 155’s Shared Drive.

Completed or unreadable recorded positions are not rerun. Missing inherited results are reported.

## Select a cohort to preview

Default: `Default / S1T`. This changes only the preview; the final cell checks the whole pipeline.

In [ ]:
from pathlib import Path
from copy import deepcopy
from concurrent.futures import ThreadPoolExecutor
from threading import Lock
from types import SimpleNamespace
from urllib.error import HTTPError, URLError
import base64
import gzip
import json
import math
import os
import pickle
import sys
import tempfile
import time
sys.dont_write_bytecode = True
local_root = Path.home() / 'DataScience/Semester4/GA-Work/hybrid_variable_framework/Dynamic_Routing_Eval_Framework'
for framework in (local_root, Path.cwd().parent, Path.cwd()):
    if (framework / 'daqr').is_dir():
        sys.path[:0] = [str(framework), str(framework / 'notebooks')]
from cohort_run_publication import RunCohort, SharedCohortStore
from daqr.config.pipeline_backup_index import GitHubBackupIndexStore
from daqr.config.tae_workflow import WorkflowStore
from daqr.evaluation.multi_run_evaluator import MultiRunEvaluator
from daqr.pipeline.recovery import load_compatibility
from evaluator_derived_results import read_completed_result
PIPELINE_ID = '155'
PIPELINE = f'Pipeline_{PIPELINE_ID}'
GIT_KEY = f'pipeline-{PIPELINE_ID}'
EXECUTION = 'p155-original-native-34fa18cab8194b50abe42e7758de66eb'
ALLOCATOR = 'ThompsonSampling'
COHORT = 'S2T'
workflow_store, cohort_store = WorkflowStore(), SharedCohortStore()
_, index, workflow = workflow_store.load(PIPELINE)
if workflow.get('activeNativeExecutionId') != EXECUTION:
    raise RuntimeError('Pipeline 155 does not point to its original full-frame execution. No work started.')
storage_store = GitHubBackupIndexStore()
binding = storage_store.read_record(storage_store.snapshot(PIPELINE_ID, read_indexes=False))['storage']
if binding['pipelineId'] != PIPELINE_ID:
    raise ValueError('Registered Shared Drive belongs to another pipeline.')
plan = workflow['plan']
if ALLOCATOR not in plan['allocators'] or COHORT not in plan['cohorts']:
    raise ValueError('Select an allocator and cohort from the registered plan.')
models = plan['models']
target = plan['runs']
replay = 'Tb' if COHORT.endswith('Tb') else 'T'
set_id = f'{ALLOCATOR}:{replay}:{COHORT[1:-len(replay)]}'
pinned = cohort_store.open_published(GIT_KEY, EXECUTION, COHORT)
PREFIX_PINS = {'S1.5T': {'branch': 'cohort/S1.5T', 'commit': '38532f24f6a80b8215fe18316f3f91ddd1d3819b', 'tag': 'versions/pipeline-154/pipeline-154/S1.5T/38532f24f6a80b8215fe18316f3f91ddd1d3819b'}, 'S1.5Tb': {'branch': 'cohort/S1.5Tb', 'commit': '1c4139b58b4e3214ed841909376c2440732fdb7b', 'tag': 'versions/pipeline-154/pipeline-154/S1.5Tb/1c4139b58b4e3214ed841909376c2440732fdb7b'}, 'S1T': {'branch': 'cohort/S1T', 'commit': 'd42ec3196d0a38b4d484b95db5e2967ef53abe72', 'tag': 'versions/pipeline-154/pipeline-154/S1T/d42ec3196d0a38b4d484b95db5e2967ef53abe72'}, 'S1Tb': {'branch': 'cohort/S1Tb', 'commit': '86d1d1e9cdc91af1a0b1052d66276609d1326a43', 'tag': 'versions/pipeline-154/pipeline-154/S1Tb/86d1d1e9cdc91af1a0b1052d66276609d1326a43'}, 'S2T': {'branch': 'cohort/S2T', 'commit': '6de24314667902c1a829df0152cf6bb36766fc6d', 'tag': 'versions/pipeline-154/pipeline-154/S2T/6de24314667902c1a829df0152cf6bb36766fc6d'}, 'S2Tb': {'branch': 'cohort/S2Tb', 'commit': 'dbdf83a4b495ddef7e679ae1b4b575925f9f00f3', 'tag': 'versions/pipeline-154/pipeline-154/S2Tb/dbdf83a4b495ddef7e679ae1b4b575925f9f00f3'}}
MIRROR_PINS = {'S1.5T': {'branch': 'cohort/S1.5T', 'commit': '7c2ecd7b9455be4470a5451a88dc82f3a028e38d', 'tag': 'versions/pipeline-159/p159-from-p155-34fa18cab8194b50/S1.5T/7c2ecd7b9455be4470a5451a88dc82f3a028e38d'}, 'S1.5Tb': {'branch': 'cohort/S1.5Tb', 'commit': '2fd771ff0e04b753566575b8888b93327a963dfc', 'tag': 'versions/pipeline-159/p159-from-p155-34fa18cab8194b50/S1.5Tb/2fd771ff0e04b753566575b8888b93327a963dfc'}, 'S1T': {'branch': 'cohort/S1T', 'commit': '6c9383636e69253e5e17a7bc2d0a497415a3ecff', 'tag': 'versions/pipeline-159/p159-from-p155-34fa18cab8194b50/S1T/6c9383636e69253e5e17a7bc2d0a497415a3ecff'}, 'S1Tb': {'branch': 'cohort/S1Tb', 'commit': '6813217e6b02e75646711cffee83bd65057aba3b', 'tag': 'versions/pipeline-159/p159-from-p155-34fa18cab8194b50/S1Tb/6813217e6b02e75646711cffee83bd65057aba3b'}, 'S2T': {'branch': 'cohort/S2T', 'commit': '7c7c4c23956bafff725aef2801d37ee285c4b9de', 'tag': 'versions/pipeline-159/p159-from-p155-34fa18cab8194b50/S2T/7c7c4c23956bafff725aef2801d37ee285c4b9de'}, 'S2Tb': {'branch': 'cohort/S2Tb', 'commit': 'dee9802da4b8ec1af9913ed4160a0075fdd7424b', 'tag': 'versions/pipeline-159/p159-from-p155-34fa18cab8194b50/S2Tb/dee9802da4b8ec1af9913ed4160a0075fdd7424b'}}
def read_historical_pin(pipeline, execution, reference, cohort):
    if cohort_store.ref('tags/' + reference['tag']) != reference['commit']:
        raise ValueError('Historical checkpoint tag differs from its pinned commit.')
    return RunCohort(cohort_store, dict(pipeline=pipeline, execution=execution, cohort=cohort),
                     reference['branch'], reference['commit'], reference)
prefix_snapshot = read_historical_pin('pipeline-154', 'pipeline-154', PREFIX_PINS[COHORT], COHORT)
mirror_snapshot = read_historical_pin('pipeline-159', 'p159-from-p155-34fa18cab8194b50', MIRROR_PINS[COHORT], COHORT)
print(f'{PIPELINE} · {ALLOCATOR} / {COHORT} · {target}-run continuation')
print(f'Workflow: {index[PIPELINE]["version"]} · Git cohort: {pinned.head}')


## Recorded constructor settings

Portable snapshot of Pipeline 155’s hash-verified original metadata and per-cohort position policies. The execution cell preserves their constructors and model settings and changes the registered target of 9 runs. Source SHA-256: `df086baf170c29b0e6f23d664b146841d75a7aff62f37dccc6eec0484310be8f`.

In [ ]:
profiles = json.loads(gzip.decompress(base64.b64decode(
    'H4sIAAAAAAACE+2da28byZWG/4rATwlgWnU5dfN+8ng02QWS2Gt7gl0sFkKdusjMUKRCUvYowfz3reZFN2skeS2REvkOMBDZ3Sx2'
    'ny4+dYyn69S/ej+WGk+Hs1cfX8neq3/14mwW0y/T+cscT2aDz6V7ncajOjg6ncTZYDya7xwOxynOxpP53mGcto/0/vOUB7PX53te'
    '9H75EidH88Y4TstwMCqHyw/OmxmdDocveuXXk+F40fIhj0en09WOT3F6eDoa1PHk+LD7fO9VjcNpedE7HowO/9F91/TwpEwOJ+PT'
    'WdspX/RGp8eHJ3H26byJbsN8d9tCL3rt74xLbqeaF5fdO992uLjEdq6/tW3jWRwuv6L3Spvf2rZFZA4Ho1kZTQezs94r8VKZ8+3t'
    'Asq1TbOzk7bpIo4v5lE4TPEkpnkDy8uZb52W7rykltQaKKPPq09PZ+PUAjEbpPb543Euw3ZG/9N7O4lp2DX5p7+WdluGP7/5ob05'
    '+K93l9++eXc6mZ4OZpe3Db7e+L8veuPPZfJlMuguYTY5bed08ulsOkgtvnESj6eLqExOR+1VeNGbpjjsAv5SdK/LKE4G42td5vJV'
    'H8fJL+PPbdvyRbst41F3zPxP+/JR1zMuffbahheXg3AlIu2kurgdjmudllk7I9Gd0Wmtg19XPeDG27vcNsiro9rl8/i8h/22jPSH'
    'MpsNRosO/HXY2sZL/bvM4jIi3Ue7i5sf2FvEbdS66fKGcpylT71rJ26F+O3a7fu95turwxrT/KcnXorWW47i8fGVbfL8JD6d8aRd'
    '5f1OwsxP4k+Pc4V63viy215peNna/Bf+VVvTWTnpfxlMy/X21Ly9wf1vy7W43RQ4ef3SWt/kJUvud5WunVUHi1WXBzm/h5znvAA3'
    'wU1wcwe4uejgoOb3UHNJBzATzAQzd4CZ17s06Pk99PyKB+AoOAqO7gBHL3dYMPR7GHqFf+An+Al+bj0/27GLRs8+fIrK2Ha0c2S5'
    'cJHORB+UZU0qWF1r0rbEaKSvPmpLKQr2prpMiaiqJIJKgZJpX3QyHg7S2QqnP7Xf5Zxrouvz3ZYPC4IoPSfI50EuozQP9cm4w1a7'
    'wt7+KHa/qHfjhrAG3HfzFg9Wh7ZrGZ9OUnlf6vxbzmblz2V0NPvUvsUG7dspxO5Nb//naZlM908aLyZHcZIGcf9lx/Xh/vRTnJT9'
    '1IL4a38Wp79M97uvPurwmPv1dHY6Kf2uRx6VvhLKSin1/qj8Ousvidyfyf0f3/93/937tz/9x58P/Mu/T9tQ0U5sFccWJ92ClZww'
    'wVqrkgkpu+hcLFqQysaoIJXRjhoLyBVFwYokqgu5HSq4N785Xew+tJ4170YtfL8MRt2wsIhO/2QZnv7iru5/ll1HmAyOBqM4XMXu'
    'w++Fyvlg6N6hytM0sELtHw+OFoNgO1Ya1Z+U1DH57CJMaXx83M6pu9P9FEd5kFtQ91ufnA2OWzvxH5P98jkOT+et7B+3YA4O297D'
    '5cbx5OXJ2eVIcvsabUw2nESu2SklOMeUvIg1FWk5ZM012OKTlNx2cyna+SLJeh2C636rq0i9veBvF99LHvWlgUmFSf22rMIgq0BW'
    'gawCJhUmFdwEN8FNmFSYVDATzAQzYVJhUsFRcBQchUmFSQU/wU/wc8dMqpIck7axSmGCqMEmNkm4YJSJiUywXosgivaVORTnhXfa'
    'VHYxxsI5h02bVNXO7xH04EqgTqXpt4gNPi88YIu079fBZDq73auqFLWR7VdvlAzCWlczV11IZKVNJu91TU5pX2KoWlqylUMMLZ5F'
    'EpdI8Kpr9aoKVhVW9VsyDIX8AvkF8gs4VThVUBPUBDVhVGFUQUwQE8SET4VPBUVBUVAUNhU2FfQEPUHPHXGpRcYc2OVKJYpsjWZF'
    'Qmmdoq9FEyuZVW7dV1ZLMnJUXpEnn21V7KWwV1zqJfiuTNUSLW9uIssd2vWCRgutOO3d4BcXM0rfnc0+jUf/HqefFi3cTzze4Nja'
    'xkm5GAo+DtrXzr/t/OB+98lJORnG7sQ+9i6Yc8XPMQrIrl3QLcaHZ/sEEMYsjFkYs+Dn1uvnAE1AE9CEngMy76nnAEwAE8CEnQM6'
    'v8POAaKAKCAKOQeA/j/kHOAJeAKe2+/mbJAkY6oUXDS1UPAp2SqMK2SKSMoV6Vy2XE3IMRifXQzKUREmEwnvv9nNXYDlDjXXfisl'
    'X3xucdSTc3N8IefkdTn3uFVJbx1rLg8tEGPXBox2TrM4OhqW4/bicHqaUplOz39uDQjiWyfPv16+3Hud27dP2yFxuPd6HrnL48xf'
    '5i9uPmg59PywTEf2/vC2tXjcjngzHuV5z5v+8aYx6e18w97FGawavDI2fTh/s/c+jvL4eO+nOBi2X870aY5VwxIno/ax864oMX49'
    '6/Hr7hv6FNzdRpi6BdYMRAVRQdStIeqDib2N8PTZKzXQFDQFTbeGpo9g/TbC1S3ybSAsCAvCbg1hH1gJboSuWyLjQFaQFWTdErLe'
    'tMJk8kErQcJS5WpVcsyKPGeWWRtjKUTPqQjhDJHMxQfHRFGXXKUpTj5QXdTaFfMc/HOO+A8nJd1WC1UHR+6RVpWclOM46GLfn55y'
    'i9x0tYuuLTjJ0vS7n3W/QeZsf/xl1D+ZjPNpmivFOXLKpP9Z7l9svWOFyqiF0pGELp6ilI509MEbwb7dHa1V9hyCSZaljKXzuIlY'
    'ucBWdwVBUUl1rZVUGaVUMVMT1QXwBBOeYMJMTczUBDQBTUATMzUxUxPABDABTMzUxExNQBQQBUQxUxMA3amZmoAn4Al43quKqmFd'
    'pS5MHIQXVkUbamlBkaS8cByL95RMaZtV4aKzE1kW56wUbH0pmKl5rYzq2SgeD9Krjw9QRXXZ1i2D1w0jlJr/EG4chxTWKUR9AIw6'
    'GHWeul3bUvBt6UKDwB6wB+x9tx/bUuht5UqBQB6QB+Q9oOHaUvht+VJ/wCAwCAw+kKPaUgRu8Vp9wB/wB/zdxzIp4YIyuipSOehM'
    'WjtjZQpR1+BcMSI4o7RSzkkv26G55uqjcjanakjHB5rft5jM927R0i1z+4wUQT/a3L751LN+7bTUatpbf7r8ra2OM/t5MRz0Z3L/'
    '/cGbt387eH/wY//D25/fvzlYzd/7avqeESnZ7FK2RXJwygenMwdN7AXH6Es3vy8LoZm8S7K4GnXykq1gzoI0pu+tY/rehSF8gFKu'
    '98gaTqaDYdfCggOLd4e5pE5gLrbdklecf3rBh9syjdueb1mM7lfGJEjHh0lCUOXgotfcq04E8h/kP8/OdwLiWyxQgXAgHAjfbncL'
    'gG+tDAa+gW/ge1c8NEC+A2IbSAfSgfRdcOrA+ZZLeqAcKAfKn/bzAWxYFVcLCxG65UAr+eo1kQ+SFZnIUTmnI0lZJKVqbVFZKmVt'
    'tEIFVx/o+YCF4l5J7MVzAgerQ295XqCdp6ZHel5gUfD3/GEAc3sBXxsKCVKaYyStmCRHp1lw90iADaI48llQMEYapUSyJGN0PhWS'
    'xDKngCcA1voEgHoOc4R3WMMvT+ph1iNAUQosG/CYKcLvneoOL8r6zOj4rPw22Ih/KuGfSjs3h3hHdTBoB9qBdjs8fXjn7SkICAKC'
    'gDs6c3inZSPIB/KBfGuXglJpoYN0RDbqQrbaEqLjWmtyNrMnaZITWgbBxjgZlWPjaszB1qqsVBtYFFQ5a/WmFgVdOULVn3/JJQv2'
    '+wuDDkYnp7P+dBaPuoavrBTan++b3u4bud2VZEtV5GISVDtzq6yuZEx23qectEk5GPJRVVGDyiKwd0pSaIeThW9cp2/k51GUGCt5'
    'Psjy53jsCRYTOQ0c5tNwmMAtcAvcArfrxu0OClSgFqgFaoHadaMW9hb4BX6BX+B3I/jdcXUM7AK7wC6wu17s3uCtqzYcrOQgdfSx'
    'sgmZrTKec86+pChtlcbqWgoxu+qjSUkZrUo339UXtwlv7bWnTXtrlmsU1+0mRe2qKi5JoUJyPqsUDWlpkvBGScdJVJWE4pJyJeVc'
    'TdHaqjUJVyTE9ZrFNWplP4Fa2c97VfhbS4Hg4TskNqhYDZTChgOkAClAirrRT75uNCAKiAKimAYMnO6uwQZYAVaAFTWUn3YNZQAV'
    'QAVQd2HSclTZO9ciTEnYYpKyynNSMSZrhDYpRKVZeEO5ap+zJp91iaQ4mVhCoE1XMnaG/FoqGfNdpYxlccyOc47GETvlI9ecZFUh'
    'lraraCFF1tbGoCoFbyJLpYi1tCHLHAoM7XoNrYKfhZ/9rjwDpVGQZSDLgJ2FnQVGgVFgFG4WbhYIBUKBUJhZmFlgFVgFVuFlgVR4'
    'WeAUOAVO729lSbmkZCo5W6mDrZqqFlkR++pYV21Z2qxqSMbnnIT1gaUWQWmVvLRWbNzKUse1dVhZdbuUdT5zNMKHyMGVZK2RzF7b'
    'FjEpSEpfWRhVHOkiqZCq85m0JNr/OrQDIGXXIGUXJRZefXyAas+Lph46x1hQ/8GTDFJPTsg+yfUsvuHJr2eYYWCNe+Qgz9DZgrRb'
    't0IuOAvOgrNPSuqCslu2Li8YC8aCsU/U+oK2W7saMLgL7oK7T1ALg7lbuQYxeAvegrfr9sY2Ku+jkFYGzcRGGuccCdaCEgdVrbcu'
    'hJBTteyi8VVoL7POhZVU2tcH8sadjZzkkv96gz++zRsbH4TdUFnnyXwc6s/k7UJZJWNLkZGSCZyjkiGRNbkyVR2EzEkX60snSL3z'
    'IkWhcyrKWVnadlschPJahfIDVGFGgrLbShnFRJB+IP2AMoYyBkfBUXAUShhKGAwFQ8FQKF8oX3AVXAVXoXTBVChd8BQ8BU+/Udlm'
    'a40RtcpadZGi2iRylIJqrMkmxaGrJ2xsstLEILkUFTyFKCmVaojMppWtdFY+5lTfcy97R/lllaOyShlZSk3Z+BBKadFUpgUsWkUm'
    '+eykaH+qkFlE4yO3HaaYXLMmBTG7TjGroGWhZVFLBAkGEgxIWUhZUBQUBUWhZKFkQVAQFASFkAVLd1nIgqqgKqgKHQsdC5qCpqDp'
    'umSsdsoQk7fJ+SyU94KNaLuCz8lbk7VWQToTQkqBWYkqU442CW/afhfKhmWs9ppow/Nn7yjIbEt2ZHNm1QJatOfsPWWvjLAi2cC1'
    'hTZXrTiqlChlwzpJJqnYtlALC027Rk3LqMiMJXJR3APFPZCcQORut8gFZoFZYBamF5DFErtALBALxEIFb4cKBnaBXWAXrhjIxRK9'
    'wC1wC9yuSSazNc6TksJobSzxfPXe4HRSLEmZpKvVUQpXoraivSIhFCXyuVBnm93GZbKzasMyme+oxhxCyZEcFSWiiFRsC60wpapY'
    'ZekmTQdJbZuNiihkKZWUVquqXcpJpYRqzGu2ySjHDJ+MyiLIP5B/wBfDFwOjwCgwCh8MHwyEAqFAKHwvYLrzvhdYBVaBVfhc+Fzg'
    'FDgFTtfna0lJrqUKZ5hYGW9skRyqTFVJH5PQkdshKhrJ7T9bvXHesjbWKi2U3PziuXbjk3/5rirNFEsK0pOTVTNnH2PVjpNoUA0q'
    'aRPYs6XILUCcha82+6BDkJSStjUThO1ahS3KNEPXovIIkg8kH5C1kLWAKCAKiELVQtUCoAAoAApRC5TurKgFVAFVQBWaFpoWMAVM'
    'AdM1SVoTPSVTfDGi5KTIWFGjj6nFPPiYUykhBRsz6yqUtG2DI24tc6rstePvkbSrc/k9dWjJhUcTsHOf2K+d4Fy5zP50+btaHWf2'
    'z0Ws2l/448XI4vdXn1nepbNrNjFnRZWUj6EGTbqrfs3OKSoi25qMKUGG0g03LZpOm5qSsFZlnVSqiRJ3Ha6OJ8enw9ZNeiuF3caO'
    'L3vjL6O9lWh0+35vedi/7S0ude9kMv57Sd3evcarsxd7o/FstW9hdPf+cRqHgzpIi0H0xWVbvpKTt4lxrbR4Ovfl/cGbt387eH/w'
    'Y//D25/fvzlYyfKvVzSWQZEvIbjoTRXFJl2qNa3Pu2JaBw9S+8KJVfDCk1FOq0olq6hYF58SXPkaXPnHT+Pjk3bjPsTjkzbYHr36'
    '+AA1s683enNOeGsadylrw4LC31n55PXy5d7r3FqetkPicO/1PCyXs7K/zF/cfNAyUfuhBadLyfb+8La1eNyOeDNu3Xv+Y/jjTRnc'
    '2/mGvYszWDV4JZP7cP5mb8H7vZ/iYNh4NUVBFeR+u+LInzA1t3T5YDATzAQzn68Sf8LE3MrFgsFL8BK83AYD/oTJueVLA4OhYCgY'
    '+tyF9xPm5xYvBAx2gp1g55P22y6EwK505YpDsMSKRSle1OxtzD6YkrQWNtnsleasAxlJ3czY6q1PMdoHmoQ8umHy8cHq0FtcK3kV'
    'Hqto9Hym8Wzan91RFFol38nsSj7GmBsMiiqFiqyRuTiuXBIrQy5QNVx18YWl806z5BRLjPCmG/KmD1AdGub0idUYQV6BvAJ5Bdwp'
    '3CmoCWqCmrCnsKcgJogJYsKfwp+CoqAoKAqDCoMKeoKeoCfo+RAONYisY7Usis1Jes9WWmsNu6SVTIXIOyuKLJlKkMY7FSwVslS1'
    '0aR93rhDlUY+ukO9q1BzN1nacI45kK7aFba6kLOcpNXJU3unkyCWmZMLsbINQZjqjfSltDhDom5Goioo1KeeVtxr2ESFkHsMM8MS'
    'J11V+vMuI5ERPOuM4O4bCiO6HUYUEAQEAcF1QxCCEwAEAAHAnQUgfCWgCCgCioAi9CNgCBgChoDhHTYxZlW0IVGtZ1lS6BYkzc5E'
    'JxWrYmRINrB3NnHuZhpyrjlLHz1bx+yt2bhNFM49pk08r3o7m6rbnaIQ3kfRLahLWbWoherbH6ejs5WstMmIUgMxiRa9VG2UzihL'
    'Kkdrla8eTnEjTpHXVtE2Dk8+xcOTNnZNljjoyHBlw63LHHzHogZXhiYs/vrI1SawqAHSDkzOBDyx5CvQCXQCnRsVmLsJzm1c6BXY'
    'BDaBzbVrz90E6HYv7wqUAqVA6Vpl6W5idHsXdQVCgVAg9D6KVcoiYjLRRymDrkaR1NV7YxIlYXzpXoeakxbsjQjVxCqTS4W1oqBp'
    '4xM2nX20hV/PJ2zyHVVvQ1fzNiQVvElGs2ctrAhZCuvZRm7RkzKm6KzVMpEmYvJJZh9sO6g6A7m6Kbm6trK30KtblV7cVooCCQYS'
    'DAhW4BOCFfAEPAFPKFYoVoAT4AQ4IVkhWQFTwBQwhWaFZgVEAVFAdGdEq9LKamItDFntSWdvpIyZjCmcfRFVlWi0LMIVEzIFCmyF'
    'caZ4JzSpunHR6oV6fNF6R2lcrdlW6207MAmTWvAk+eSjdN5a114l5Slr6bMwMpAzmTjZ4ttx0igRYFo3ZFoVPCs86zfnF6iogewC'
    '2QUsKywr0Al0Ap1wrHCswCawCWzCsMKwAqVAKVAKvwq/uq1+FQgFQoHQ+9jVboKqF0UGrwIRm8ii+pgoG6GTl6YUitpllkoWUaKw'
    '0WTvUpTKOq9i3YlprHfUCE46VZJG2mBq8MF7LSSz60JbTVApBfJCOW1lTjkGW4JoO4OSnGoJAdNYH1+u/vZ/mA/fhr3BAgA='
)))
profile = profiles[set_id]
policy = profile['policy']
threats = list(plan['threats'])


## Verify and restore the pinned results

Use Pipeline 155’s published checkpoints first. Runs 1–8 can also be read from Pipeline 154’s pinned predecessor. For run 9, the immutable migration snapshots are used only for exact copies of Pipeline 155’s own original checkpoints; the live Pipeline 159 workflow and results are not used. No evaluator state is loaded.

In [ ]:
def frames_at(position):
    return policy['baseFrames'] + (position - 1) * policy['frameStep']
def checkpoint_source(key, position):
    if key in pinned.experiments:
        return pinned
    if position < target and key in prefix_snapshot.experiments:
        return prefix_snapshot
    if position == target:
        copied = mirror_snapshot.experiments.get(key, {})
        if copied.get('state') and copied.get('source', {}).get('pipeline') == PIPELINE:
            return mirror_snapshot
    return None
legacy_runner_lock = Lock()

def decode_saved_runner(data):
    with legacy_runner_lock:
        try:
            return pickle.loads(data)
        except ModuleNotFoundError as error:
            if error.name not in ('original_legacy_allocator', 'original_legacy_environment'):
                raise
            runtime = Path.home() / '.local/share/dsci602/migrations/p152-recovery-20261113/common-base-candidate/runtime'
            load_compatibility(runtime, 'a9986c7a4a55f1dead4e9b86f4522e71fceeda0315eeb0ea9337b9f005303516')
            return pickle.loads(data)

def read_saved_checkpoint(snapshot, key, entry, threat, position):
    for attempt in range(3):
        try:
            if entry.get('state'):
                return decode_saved_runner(snapshot.read(key, 'state'))
            source_pipeline = entry.get('source', {}).get('pipeline') or f"Pipeline_{snapshot.identity['pipeline'].split('-')[-1]}"
            return read_completed_result(snapshot, key, pipeline=source_pipeline,
                set_id=set_id, threat=threat, position=position, frames=frames_at(position), models=models)
        except HTTPError as error:
            if attempt == 2 or error.code not in (429, 500, 502, 503, 504):
                raise
        except (TimeoutError, URLError, ConnectionError):
            if attempt == 2:
                raise
        except ValueError as error:
            if attempt == 2 or not str(error).startswith('Checkpoint content differs:'):
                raise
        time.sleep(2 ** attempt)
def resolve_saved(scope):
    threat, position = scope
    key = f'{ALLOCATOR}/{threat}/{position}'
    snapshot = checkpoint_source(key, position)
    if snapshot is None:
        return threat, position, None, None
    entry = snapshot.experiments[key]
    try:
        checkpoint = read_saved_checkpoint(snapshot, key, entry, threat, position)
        if entry.get('state'):
            if (checkpoint['id'] != position or not checkpoint['is_complete']
                    or checkpoint['frames_count'] != frames_at(position)):
                raise ValueError('Runner identity, completion or frame count differs')
            row = dict(results=checkpoint['results'], winner=checkpoint.get('winner'), exp_id=position)
        else:
            row = checkpoint
        if set(row['results']) != set(models) or any(
                result.get('error') or result['frames_count'] != frames_at(position)
                or not math.isfinite(float(result['final_reward'])) for result in row['results'].values()):
            raise ValueError('Saved model results are incomplete or invalid')
        return threat, position, row, None
    except Exception as error:
        return threat, position, None, f'{type(error).__name__}: {error}'


def restore_selected_cohort():
    global saved, read_errors, ready
    saved = {threat: {} for threat in threats}
    read_errors = {}
    with ThreadPoolExecutor(max_workers=4) as readers:
        for threat, position, row, error in readers.map(
            resolve_saved, ((threat, position) for threat in threats for position in range(1, target + 1))
        ):
            if row is not None:
                saved[threat][position] = row
            elif error:
                read_errors[(threat, position)] = error
                print(f'Checkpoint unreadable — {threat}/{position}: {error}')
    ready = []
    for threat, rows in saved.items():
        missing = sorted(set(range(1, target)) - rows.keys())
        existing_target = checkpoint_source(f'{ALLOCATOR}/{threat}/{target}', target) is not None
        recorded_target = (workflow['allocators'][ALLOCATOR]['cohorts'][COHORT]['threats'][threat]
            ['experiments'].get(str(target), {}).get('state') == 'done')
        if target in rows:
            print(f'{threat}: {len(rows)}/{target} verified; run {target} already saved')
        elif missing:
            print(f'{threat}: {len(rows)}/{target} verified; inherited positions missing: {missing}. Skipped.')
        elif existing_target or recorded_target:
            print(f'{threat}: run {target} already recorded but unreadable; skipped without rerun')
        else:
            ready.append(threat)
            print(f'{threat}: {len(rows)}/{target} verified; only run {target} remains')


restore_selected_cohort()


## Saved results and winners

Uses the existing `display_run_results()` formatter. Call `read_saved_experiment_log(threat, position)` for an exact Git log when one exists.

In [ ]:
def display_saved_pipeline_results():
    for threat, rows in saved.items():
        original = profile['attacks'][threat]['configuration']
        for position, experiment in sorted(rows.items()):
            frames = frames_at(position)
            capacity = policy['baseFrames'] if original['base_capacity'] else frames
            oracle = experiment['results']['Oracle']
            environment = oracle.get('env_type') or experiment.get('attack_category') or 'not recorded'
            attack_rate = 0.0 if threat == 'none' else oracle.get('attack_rate', 'not recorded')
            context = SimpleNamespace(frames_count=frames, configs=SimpleNamespace(
                allocator=ALLOCATOR, environment=environment, attack_strategy=threat, attack_rate=attack_rate))
            row = dict(experiment, winner=experiment.get('winner') or 'NA')
            print(f'[{ALLOCATOR}] {threat}: ⏩ SKIPPING EXPERIMENT {position}: ALREADY COMPLETED AND STORED')
            MultiRunEvaluator.display_run_results(context, position, {position: row}, capacity * original['scale'])
def read_saved_experiment_log(threat, position):
    key = f'{ALLOCATOR}/{threat}/{position}'
    snapshot = checkpoint_source(key, position)
    if snapshot is None:
        return None
    data = snapshot.read(key, 'log')
    return data.decode('utf-8') if data is not None else None
display_saved_pipeline_results()


## Prepare the native execution cell

Definitions only. Runner/model states and logs publish to Pipeline 155’s Git cohorts. Evaluator checkpoints publish to its registered Shared Drive. Verified files are removed from temporary staging after publication.

In [ ]:
from contextlib import redirect_stderr, redirect_stdout
import fcntl
import shutil
from daqr.config import tae_workflow
from daqr.config.experiment_config import ExperimentConfiguration
from daqr.config.gd_backup_manager import GoogleDriveBackupManager
from daqr.config.local_backup_manager import TeeStream
from daqr.core import qubit_allocator
from p155_causal_history import CausalExperimentConfiguration
from p155_remote_archive import ExperimentArchive
from p159_worker_handoff import model_worker_handoff, report_status
from p159_staged_continuation import NotebookLog, optional_log
from tae_background_status import BackgroundWorkflowStatus, reconcile_index
class GitExperimentArchive(ExperimentArchive):
    backup_pending = False
    log_path = None
    def _handoff(self, key, checkpoint, models, evaluator, *, log=None):
        super()._handoff(key, checkpoint, models, evaluator, log=self.log_path)
        if self.log_path is not None:
            self.log_backup_pending = not self.remove_verified_log(key, self.log_path)
    def finish(self, position, results, evaluator_checkpoint):
        try:
            return super().finish(position, results, evaluator_checkpoint)
        except Exception as error:
            self.backup_pending = True
            print(f'{self.key(position)}: Backup pending — {error}', flush=True)
def native_configuration(threat, staging, drive):
    original = profile['attacks'][threat]
    configuration = deepcopy(original['configuration'])
    allocator = configuration.pop('allocator')
    physics = configuration.get('physics_params') or {}
    if 'entanglement_success_factor' in physics:
        factor = physics.pop('entanglement_success_factor')
        configuration['testbed_config'] = dict(configuration.get('testbed_config') or {},
            entanglement_success_factor=factor)
    configuration.update(runs=target, resume=False, use_last_backup=False, overwrite=False,
        pipeline_storage=binding, staging_dir=staging, drive_service=drive,
        scenarios={threat: configuration['scenarios'][threat]})
    causal = threat in ('adaptive', 'onlineadaptive')
    configuration_class = CausalExperimentConfiguration if causal else ExperimentConfiguration
    config = configuration_class(
        allocator=getattr(qubit_allocator, allocator['class'])(**allocator['kwargs']), **configuration)
    for name, model in original['modelSettings'].items():
        for field in ('kwargs', 'seed_offset', 'runner_type'):
            config.algorithm_configs[name][field] = deepcopy(model[field])
    if causal:
        config.bind_causal_models()
    config.resume_allow_drive_downloads = False
    config.remote_experiment_archive = True
    return config
def run_selected_cohort():
    if not ready:
        print('No eligible new runs. Saved and missing inherited positions were not rerun.')
        return
    with (Path(tempfile.gettempdir()) / 'pipeline-155-notebook.lock').open('a') as lock:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        _, fresh_index, fresh_workflow = workflow_store.load(PIPELINE)
        heartbeat = fresh_index[PIPELINE].get('runnerStatus') or {}
        is_live = (heartbeat.get('state') == 'running'
                   and time.time() - heartbeat.get('updated', 0) < heartbeat.get('heartbeatTimeout', 180))
        if is_live:
            raise RuntimeError('Pipeline 155 already has an active runner. No work started.')
        if (fresh_workflow.get('activeNativeExecutionId') != EXECUTION
                or fresh_index[PIPELINE]['version'] != index[PIPELINE]['version']):
            raise RuntimeError('Workflow changed; refresh the restore cells before running.')
        fresh = cohort_store.open_published(GIT_KEY, EXECUTION, COHORT)
        if fresh.head != pinned.head:
            raise RuntimeError('Cohort changed; refresh the restore cells before running.')
        run = cohort_store.open_run(GIT_KEY, EXECUTION, COHORT)
        if run.head != fresh.head:
            raise RuntimeError('Cohort has unpublished work; no experiments started.')
        latest_storage = storage_store.read_record(storage_store.snapshot(PIPELINE_ID, read_indexes=False))['storage']
        if latest_storage != binding:
            raise RuntimeError('Registered evaluator storage changed; refresh the restore cells.')
        drive = GoogleDriveBackupManager.pipeline_creation_drive_service()
        drive.drives().get(driveId=binding['driveId'], fields='id').execute()
        os.environ['DAQR_MODEL_MAX_WORKERS'] = '4'
        reporter = report_status(BackgroundWorkflowStatus, workflow_store, PIPELINE, EXECUTION)
        published = set()
        other_cohorts_complete = all(
            fresh_workflow['allocators'][allocator]['cohorts'][cohort]['threats'][threat]
                ['experiments'].get(str(target), {}).get('state') == 'done'
            for allocator in plan['allocators'] for cohort in plan['cohorts']
            if (allocator, cohort) != (ALLOCATOR, COHORT) for threat in threats)
        try:
            report_status(getattr(reporter, 'start', None), ALLOCATOR, COHORT)
            for threat in ready:
                staging = Path(tempfile.mkdtemp(prefix='pipeline-155-native-')).resolve()
                evaluator = None
                retain = True
                try:
                    config = native_configuration(threat, staging, drive)
                    evaluator = MultiRunEvaluator(configs=config, base_frames=policy['baseFrames'],
                        frame_step=policy['frameStep'], base_seed=policy['baseSeed'], attack_type=threat)
                    if config.get_environment() is None:
                        print(f'{ALLOCATOR}/{COHORT}/{threat}: native environment unavailable; left pending.')
                        retain = False
                        continue
                    evaluator.env_experiments = {threat: deepcopy(saved[threat])}
                    archive = GitExperimentArchive(config, run, staging, drive_factory=lambda: drive)
                    evaluator._experiment_archive = archive
                    with optional_log(tempfile.NamedTemporaryFile, mode='w', encoding='utf-8', dir=staging,
                            prefix='experiment-log-', suffix='.log', delete=False) as output:
                        archive.log_path = Path(output.name) if output else None
                        with (redirect_stdout(NotebookLog(sys.stdout, output)),
                              redirect_stderr(NotebookLog(sys.stderr, output)), model_worker_handoff()):
                            try:
                                monitor = tae_workflow.current_tracker(config)
                                monitor.set_runtime(config, phase='experiments:' + threat)
                            except Exception as error:
                                print(f'Live status pending: {error}', flush=True)
                            evaluator.run_experiment(exp_no=target - 1,
                                attack_category=config.category_map.get(threat, 'Unknown'))
                    key = f'{ALLOCATOR}/{threat}/{target}'
                    retain = archive.backup_pending or key not in run.experiments
                    if not retain:
                        published.add(threat)
                        try:
                            monitor = tae_workflow.current_tracker(config)
                            monitor.set_runtime(config, phase='checkpoint-verified', state='stopped')
                            tae_workflow.sync_completed_status(config)
                        except Exception as error:
                            print(f'Workflow detail update pending: {error}', flush=True)
                finally:
                    if evaluator is not None:
                        evaluator.cleanup(verbose=False, cooldown_seconds=0)
                    if retain or getattr(archive, 'log_backup_pending', False):
                        print(f'Temporary staging retained for inspection: {staging}', flush=True)
                    else:
                        shutil.rmtree(staging)
            cohort_complete = all(target in saved[threat] or threat in published for threat in threats)
            report_status(getattr(reporter, 'finish_cohort', None), complete=cohort_complete)
        finally:
            report_status(getattr(reporter, 'finish_run', None),
                          success=other_cohorts_complete and len(published) == len(ready))
            try:
                if getattr(reporter, 'pending', None) is not None:
                    reporter.pending.result(timeout=90)
                reconcile_dashboard_progress()
            except Exception as error:
                print(f'Dashboard status pending: {error}', flush=True)
        print(f'Published new run {target} for {len(published)}/{len(ready)} eligible threats.')


def reconcile_dashboard_progress():
    cohort_index_store = GitHubBackupIndexStore()
    cohort_index_store.repository = cohort_store.repository
    return reconcile_index(workflow_store, cohort_index_store, pipeline=PIPELINE, apply=True)


def plan_remaining_scopes(document, snapshots, mirrored, prior):
    plan = document['plan']
    target_position = plan['runs']
    candidates, unavailable = {}, []
    for allocator in plan['allocators']:
        for cohort in plan['cohorts']:
            own_rows = snapshots[cohort]
            for threat in plan['threats']:
                prefix = f'{allocator}/{threat}/'
                key = prefix + str(target_position)
                saved = own_rows.get(key)
                copied = mirrored[cohort].get(key, {})
                if copied.get('source', {}).get('pipeline') != PIPELINE:
                    copied = {}
                if saved or copied.get('state'):
                    continue
                position = (document['allocators'].get(allocator, {}).get('cohorts', {}).get(cohort, {})
                    .get('threats', {}).get(threat, {}).get('experiments', {}).get(str(target_position), {}))
                if position.get('state') == 'done':
                    unavailable.append((allocator, cohort, threat, 'recorded complete; checkpoint unavailable'))
                    continue
                if not all(any(row.get(kind) for kind in ('state', 'recoveredState'))
                           for earlier in range(1, target_position)
                           for row in [own_rows.get(prefix + str(earlier)) or prior[cohort].get(prefix + str(earlier), {})]):
                    unavailable.append((allocator, cohort, threat, 'inherited checkpoint unavailable'))
                    continue
                candidates.setdefault((allocator, cohort), []).append(threat)
    return candidates, unavailable


def pending_pipeline_scopes(document):
    all_cohorts = document['plan']['cohorts']
    own_pins = cohort_store.pipeline_cohorts(GIT_KEY, EXECUTION)
    snapshots, mirrored, prior = {}, {}, {}
    for cohort in all_cohorts:
        snapshots[cohort] = read_historical_pin(GIT_KEY, EXECUTION, own_pins[cohort], cohort).experiments
        mirrored[cohort] = read_historical_pin('pipeline-159', 'p159-from-p155-34fa18cab8194b50',
            MIRROR_PINS[cohort], cohort).experiments
        prior[cohort] = read_historical_pin('pipeline-154', 'pipeline-154', PREFIX_PINS[cohort], cohort).experiments
    return plan_remaining_scopes(document, snapshots, mirrored, prior)


def select_pipeline_scope(allocator, cohort):
    global ALLOCATOR, COHORT, set_id, profile, policy, pinned, prefix_snapshot, mirror_snapshot, index, workflow
    _, index, workflow = workflow_store.load(PIPELINE)
    if workflow.get('activeNativeExecutionId') != EXECUTION or workflow['plan'] != plan:
        raise RuntimeError('Pipeline execution or plan changed; rerun setup before continuing.')
    ALLOCATOR, COHORT = allocator, cohort
    replay = 'Tb' if cohort.endswith('Tb') else 'T'
    set_id = f'{allocator}:{replay}:{cohort[1:-len(replay)]}'
    profile = profiles[set_id]
    policy = profile['policy']
    pinned = cohort_store.open_published(GIT_KEY, EXECUTION, cohort)
    prefix_snapshot = read_historical_pin('pipeline-154', 'pipeline-154', PREFIX_PINS[cohort], cohort)
    mirror_snapshot = read_historical_pin('pipeline-159', 'p159-from-p155-34fa18cab8194b50',
        MIRROR_PINS[cohort], cohort)


def run_remaining_pipeline():
    _, _, document = workflow_store.load(PIPELINE)
    if document.get('activeNativeExecutionId') != EXECUTION or document['plan'] != plan:
        raise RuntimeError('Pipeline execution or plan changed; rerun setup before continuing.')
    candidates, unavailable = pending_pipeline_scopes(document)
    for allocator, cohort, threat, reason in unavailable:
        print(f'{allocator}/{cohort}/{threat}/{target}: {reason}; skipped without rerun.')
    print(f'{sum(map(len, candidates.values()))} candidate run-{target} positions across {len(candidates)} allocator/cohorts.')
    if not candidates:
        try:
            reconcile_dashboard_progress()
        except Exception as error:
            print(f'Dashboard status pending: {error}', flush=True)
    for allocator, cohort in candidates:
        print(f'\n{PIPELINE} · {allocator} / {cohort}', flush=True)
        select_pipeline_scope(allocator, cohort)
        restore_selected_cohort()
        display_saved_pipeline_results()
        run_selected_cohort()


## Resume ThompsonSampling S2T

This cell resumes only the selected cohort. It starts only missing ninth positions with all eight inherited results verified.

In [ ]:
run_selected_cohort()
